# Task 5: Auto Tagging Support Tickets Using LLM

**Objective:** Automatically tag support tickets into categories using a large language model (LLM).


In [1]:
!pip install -q transformers datasets scikit-learn torch pandas

## 1. Dataset Preparation
We'll create a synthetic dataset of support tickets and their corresponding tags.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = [
    {"text": "My internet connection drops every 5 minutes.", "tag": "Network Issue"},
    {"text": "I forgot my password and cannot log in.", "tag": "Account Access"},
    {"text": "The billing amount is incorrect on my last invoice.", "tag": "Billing"},
    {"text": "How do I upgrade my subscription plan?", "tag": "Account Management"},
    {"text": "The software crashes when I try to export to PDF.", "tag": "Software Bug"},
    {"text": "I haven't received my refund yet.", "tag": "Billing"},
    {"text": "Can I use this app on my Mac?", "tag": "Product Inquiry"},
    {"text": "My router is flashing red.", "tag": "Network Issue"},
    {"text": "Please delete my account.", "tag": "Account Management"},
    {"text": "The app is very slow today.", "tag": "Software Bug"},
    {"text": "I need help setting up my new device.", "tag": "Technical Support"},
    {"text": "Where can I find the user manual?", "tag": "Product Inquiry"},
    {"text": "My payment was declined but I have funds.", "tag": "Billing"},
    {"text": "The screen goes black randomly.", "tag": "Hardware Issue"}
]

df = pd.DataFrame(data)
candidate_labels = df['tag'].unique().tolist()
print("Candidate Tags:", candidate_labels)

# Split for fine-tuning vs testing
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

print(f"\nTraining samples: {len(train_df)}")
print(f"Testing samples: {len(test_df)}")

Candidate Tags: ['Network Issue', 'Account Access', 'Billing', 'Account Management', 'Software Bug', 'Product Inquiry', 'Technical Support', 'Hardware Issue']

Training samples: 9
Testing samples: 5


## 2. Zero-Shot Classification
Using a pre-trained zero-shot classification model (`facebook/bart-large-mnli`).

In [3]:
from transformers import pipeline

# Initialize zero-shot pipeline
zero_shot_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

def zero_shot_predict(text, labels, top_k=3):
    result = zero_shot_classifier(text, labels, multi_label=False)
    return list(zip(result['labels'][:top_k], result['scores'][:top_k]))

print("--- Zero-Shot Predictions ---")
for idx, row in test_df.head(3).iterrows():
    preds = zero_shot_predict(row['text'], candidate_labels)
    print(f"\nTicket: {row['text']}")
    print(f"True Tag: {row['tag']}")
    print("Top 3 Predicted Tags:")
    for label, score in preds:
        print(f"  - {label} ({score:.4f})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

--- Zero-Shot Predictions ---

Ticket: The app is very slow today.
True Tag: Software Bug
Top 3 Predicted Tags:
  - Software Bug (0.2862)
  - Product Inquiry (0.1491)
  - Technical Support (0.1167)

Ticket: Where can I find the user manual?
True Tag: Product Inquiry
Top 3 Predicted Tags:
  - Product Inquiry (0.3800)
  - Technical Support (0.1332)
  - Billing (0.1012)

Ticket: My internet connection drops every 5 minutes.
True Tag: Network Issue
Top 3 Predicted Tags:
  - Network Issue (0.5395)
  - Account Access (0.1212)
  - Technical Support (0.1062)


## 3. Few-Shot Learning (Prompt Engineering)
Using a text generation model with a few-shot prompt to classify the tickets. We'll use `google/flan-t5-small` for good instruction-following capabilities.

In [4]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/flan-t5-small" # Using small for speed, can scale up to base/large
tokenizer = AutoTokenizer.from_pretrained(model_name)
few_shot_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Create a few-shot prompt
few_shot_examples = "\n".join([f"Ticket: {row['text']}\nTag: {row['tag']}" for _, row in train_df.head(3).iterrows()])

def few_shot_predict(text):
    prompt = f"Classify the support ticket into one of these categories: {', '.join(candidate_labels)}.\n\nExamples:\n{few_shot_examples}\n\nTicket: {text}\nTag:"
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = few_shot_model.generate(**inputs, max_new_tokens=10)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- Few-Shot Predictions ---")
for idx, row in test_df.head(3).iterrows():
    pred = few_shot_predict(row['text'])
    print(f"\nTicket: {row['text']}")
    print(f"True Tag: {row['tag']}")
    print(f"Predicted Tag: {pred}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

--- Few-Shot Predictions ---

Ticket: The app is very slow today.
True Tag: Software Bug
Predicted Tag: Technical Support

Ticket: Where can I find the user manual?
True Tag: Product Inquiry
Predicted Tag: Technical Support

Ticket: My internet connection drops every 5 minutes.
True Tag: Network Issue
Predicted Tag: Technical Support


## 4. Fine-Tuning a Model
We will fine-tune a small BERT model for Sequence Classification. To output the top 3 most probable tags, we'll apply softmax to the logits.

In [5]:
import torch
import warnings
warnings.filterwarnings('ignore')
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoTokenizer
from datasets import Dataset

# Prepare dataset for fine-tuning
tag2id = {tag: i for i, tag in enumerate(candidate_labels)}
id2tag = {i: tag for tag, i in tag2id.items()}

train_df_copy = train_df.copy()
test_df_copy = test_df.copy()
train_df_copy['label'] = train_df_copy['tag'].map(tag2id)
test_df_copy['label'] = test_df_copy['tag'].map(tag2id)

train_dataset = Dataset.from_pandas(train_df_copy[['text', 'label']])
test_dataset = Dataset.from_pandas(test_df_copy[['text', 'label']])

ft_model_name = "distilbert-base-uncased"
ft_tokenizer = AutoTokenizer.from_pretrained(ft_model_name)

def tokenize_function(examples):
    return ft_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

ft_model = AutoModelForSequenceClassification.from_pretrained(
    ft_model_name, num_labels=len(candidate_labels), id2label=id2tag, label2id=tag2id
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=5,
    report_to="none"
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

# Fine-tune the model
print("Starting fine-tuning...")
trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

...

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting fine-tuning...


Step,Training Loss
5,2.069806


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9, training_loss=2.0540733337402344, metrics={'train_runtime': 30.5551, 'train_samples_per_second': 0.884, 'train_steps_per_second': 0.295, 'total_flos': 447125308416.0, 'train_loss': 2.0540733337402344, 'epoch': 3.0})

## 5. Fine-Tuned Performance & Top 3 Tags
We'll use the fine-tuned model to predict the top 3 tags for our test set.

In [6]:
import torch.nn.functional as F

def get_top_k_predictions(text, model, tokenizer, k=3):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=-1)[0]
    top_k_probs, top_k_indices = torch.topk(probs, k)

    results = []
    for prob, idx in zip(top_k_probs, top_k_indices):
        results.append((id2tag[idx.item()], prob.item()))
    return results

print("--- Fine-Tuned Predictions ---")
for idx, row in test_df.head(5).iterrows():
    preds = get_top_k_predictions(row['text'], ft_model, ft_tokenizer, k=3)
    print(f"\nTicket: {row['text']}")
    print(f"True Tag: {row['tag']}")
    print("Top 3 Predicted Tags:")
    for label, score in preds:
        print(f"  - {label} ({score:.4f})")

--- Fine-Tuned Predictions ---

Ticket: The app is very slow today.
True Tag: Software Bug
Top 3 Predicted Tags:
  - Technical Support (0.1435)
  - Product Inquiry (0.1316)
  - Hardware Issue (0.1285)

Ticket: Where can I find the user manual?
True Tag: Product Inquiry
Top 3 Predicted Tags:
  - Account Management (0.1454)
  - Network Issue (0.1320)
  - Technical Support (0.1297)

Ticket: My internet connection drops every 5 minutes.
True Tag: Network Issue
Top 3 Predicted Tags:
  - Network Issue (0.1395)
  - Account Management (0.1363)
  - Technical Support (0.1340)

Ticket: My payment was declined but I have funds.
True Tag: Billing
Top 3 Predicted Tags:
  - Network Issue (0.1472)
  - Technical Support (0.1403)
  - Software Bug (0.1333)

Ticket: I haven't received my refund yet.
True Tag: Billing
Top 3 Predicted Tags:
  - Account Management (0.1372)
  - Software Bug (0.1359)
  - Technical Support (0.1314)
